# Feature Engineering for Trading Models

This notebook demonstrates feature engineering techniques for quantitative trading:
- **Technical Indicators**: RSI, MACD, Bollinger Bands
- **Liquidity Features**: Spreads, depth, Amihud measure
- **Microstructure Features**: Order flow, trade imbalance
- **Volume Profile**: Volume patterns and VWAP
- **Time-based Features**: Lags, rolling statistics

These features can be used for price prediction, execution optimization, and risk management.

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Import custom modules
from utils import generate_price_series, generate_order_book, generate_trade_data
from liquidity import calculate_bid_ask_spread, calculate_amihud_illiquidity

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('✓ Libraries loaded successfully')

## 1. Generate Synthetic Market Data

Create realistic high-frequency market data for feature engineering.

In [ ]:
# Generate data
np.random.seed(42)
n_periods = 2000

# Price series
prices = generate_price_series(
    n_periods=n_periods,
    initial_price=100.0,
    mu=0.0001,
    sigma=0.015,
    seed=42
)

# Order book
order_book = generate_order_book(
    prices=prices,
    spread_bps=5.0,
    depth_shares=10000,
    levels=5
)

# Trade data
trade_data = generate_trade_data(
    prices=prices,
    avg_volume=2000,
    volume_std=1000,
    seed=42
)

print(f'Generated {len(prices)} periods of data')
print(f'Price range: ${prices.min():.2f} - ${prices.max():.2f}')
print(f'\nData preview:')
print(order_book.head())

## 2. Technical Indicators

Calculate common technical indicators used in trading strategies.

In [ ]:
def calculate_rsi(prices, window=14):
    """Calculate Relative Strength Index."""
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_macd(prices, fast=12, slow=26, signal=9):
    """Calculate MACD (Moving Average Convergence Divergence)."""
    ema_fast = prices.ewm(span=fast).mean()
    ema_slow = prices.ewm(span=slow).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

def calculate_bollinger_bands(prices, window=20, num_std=2):
    """Calculate Bollinger Bands."""
    sma = prices.rolling(window=window).mean()
    std = prices.rolling(window=window).std()
    upper_band = sma + (std * num_std)
    lower_band = sma - (std * num_std)
    return upper_band, sma, lower_band

# Calculate indicators
rsi = calculate_rsi(prices, window=14)
macd_line, signal_line, macd_histogram = calculate_macd(prices)
bb_upper, bb_middle, bb_lower = calculate_bollinger_bands(prices, window=20)

print('Technical Indicators Calculated:')
print('=' * 60)
print(f'RSI range: {rsi.min():.2f} - {rsi.max():.2f}')
print(f'MACD range: {macd_line.min():.4f} - {macd_line.max():.4f}')
print(f'Bollinger Band width: ${(bb_upper - bb_lower).mean():.2f}')

In [ ]:
# Visualize technical indicators
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Price with Bollinger Bands
axes[0].plot(prices.index, prices, label='Price', linewidth=1.5)
axes[0].plot(bb_upper.index, bb_upper, 'r--', alpha=0.7, label='Upper Band')
axes[0].plot(bb_middle.index, bb_middle, 'g--', alpha=0.7, label='Middle (SMA)')
axes[0].plot(bb_lower.index, bb_lower, 'r--', alpha=0.7, label='Lower Band')
axes[0].fill_between(bb_upper.index, bb_lower, bb_upper, alpha=0.1)
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Price with Bollinger Bands')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(rsi.index, rsi, linewidth=1.5, color='purple')
axes[1].axhline(70, color='r', linestyle='--', alpha=0.5, label='Overbought')
axes[1].axhline(30, color='g', linestyle='--', alpha=0.5, label='Oversold')
axes[1].fill_between(rsi.index, 30, 70, alpha=0.1)
axes[1].set_ylabel('RSI')
axes[1].set_title('Relative Strength Index (RSI)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# MACD
axes[2].plot(macd_line.index, macd_line, label='MACD', linewidth=1.5)
axes[2].plot(signal_line.index, signal_line, label='Signal', linewidth=1.5)
axes[2].bar(macd_histogram.index, macd_histogram, alpha=0.3, label='Histogram')
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_ylabel('MACD')
axes[2].set_xlabel('Time')
axes[2].set_title('MACD (Moving Average Convergence Divergence)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('✓ Technical indicators visualization complete')

## 3. Liquidity Features

Calculate features that capture market liquidity conditions.

In [ ]:
# Bid-ask spread features
spread_bps = calculate_bid_ask_spread(
    order_book['bid_price'],
    order_book['ask_price'],
    spread_type='bps'
)

# Depth features
total_depth = order_book['bid_size'] + order_book['ask_size']
depth_imbalance = (order_book['bid_size'] - order_book['ask_size']) / total_depth

# Quote slope (how quickly depth accumulates)
quote_slope = total_depth / spread_bps

# Amihud illiquidity
returns = prices.pct_change().dropna()
amihud = calculate_amihud_illiquidity(
    returns.reindex(trade_data.index, method='nearest'),
    trade_data['volume'],
    window=50
)

print('Liquidity Features Calculated:')
print('=' * 60)
print(f'Average spread: {spread_bps.mean():.2f} bps')
print(f'Average depth: {total_depth.mean():,.0f} shares')
print(f'Depth imbalance range: {depth_imbalance.min():.3f} to {depth_imbalance.max():.3f}')
print(f'Average Amihud: {amihud.mean():.8f}')

In [ ]:
# Visualize liquidity features
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Spread dynamics
axes[0].plot(spread_bps.index, spread_bps, linewidth=1, alpha=0.7)
axes[0].axhline(spread_bps.mean(), color='r', linestyle='--', 
                label=f'Mean: {spread_bps.mean():.2f} bps')
axes[0].set_ylabel('Spread (bps)')
axes[0].set_title('Bid-Ask Spread Evolution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Depth imbalance
axes[1].plot(depth_imbalance.index, depth_imbalance, linewidth=1, color='green')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].fill_between(depth_imbalance.index, 0, depth_imbalance, 
                      where=(depth_imbalance > 0), alpha=0.3, color='green', label='Bid heavy')
axes[1].fill_between(depth_imbalance.index, 0, depth_imbalance, 
                      where=(depth_imbalance < 0), alpha=0.3, color='red', label='Ask heavy')
axes[1].set_ylabel('Depth Imbalance')
axes[1].set_title('Order Book Depth Imbalance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Amihud illiquidity
axes[2].plot(amihud.index, amihud, linewidth=1, alpha=0.7, color='purple')
axes[2].axhline(amihud.mean(), color='r', linestyle='--', 
                label=f'Mean: {amihud.mean():.8f}')
axes[2].set_ylabel('Amihud Illiquidity')
axes[2].set_xlabel('Time')
axes[2].set_title('Amihud Illiquidity Measure (Rolling 50-period)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Microstructure Features

Extract features from order flow and trade microstructure.

In [ ]:
# Order flow imbalance
order_flow = trade_data['direction'] * trade_data['volume']
order_flow_imbalance = order_flow.rolling(window=20).sum()

# Trade intensity
trade_intensity = trade_data['volume'].rolling(window=20).sum()

# Price impact (price change per unit volume)
price_change = prices.diff().reindex(trade_data.index, method='nearest')
price_impact = price_change / trade_data['volume']
price_impact_rolling = price_impact.rolling(window=20).mean()

# Effective spread (difference between trade price and mid)
mid_price = order_book['mid_price'].reindex(trade_data.index, method='nearest')
effective_spread = 2 * np.abs(trade_data['price'] - mid_price)
effective_spread_pct = effective_spread / mid_price * 100

# Trade size distribution
trade_size_mean = trade_data['volume'].rolling(window=50).mean()
trade_size_std = trade_data['volume'].rolling(window=50).std()

print('Microstructure Features Calculated:')
print('=' * 60)
print(f'Order flow imbalance range: {order_flow_imbalance.min():,.0f} to {order_flow_imbalance.max():,.0f}')
print(f'Average trade intensity: {trade_intensity.mean():,.0f} shares per 20 periods')
print(f'Average price impact: {price_impact_rolling.mean():.6f}')
print(f'Average effective spread: {effective_spread_pct.mean():.4f}%')

In [ ]:
# Visualize microstructure features
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Order flow imbalance
axes[0].plot(order_flow_imbalance.index, order_flow_imbalance, linewidth=1)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].fill_between(order_flow_imbalance.index, 0, order_flow_imbalance,
                      where=(order_flow_imbalance > 0), alpha=0.3, color='green', label='Buy pressure')
axes[0].fill_between(order_flow_imbalance.index, 0, order_flow_imbalance,
                      where=(order_flow_imbalance < 0), alpha=0.3, color='red', label='Sell pressure')
axes[0].set_ylabel('Order Flow Imbalance')
axes[0].set_title('Order Flow Imbalance (20-period rolling)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Trade intensity
axes[1].plot(trade_intensity.index, trade_intensity, linewidth=1, color='blue')
axes[1].fill_between(trade_intensity.index, 0, trade_intensity, alpha=0.3)
axes[1].set_ylabel('Trade Intensity (shares)')
axes[1].set_title('Trading Intensity (20-period volume sum)')
axes[1].grid(True, alpha=0.3)

# Effective spread
axes[2].plot(effective_spread_pct.index, effective_spread_pct, linewidth=1, alpha=0.7, color='orange')
axes[2].axhline(effective_spread_pct.mean(), color='r', linestyle='--',
                label=f'Mean: {effective_spread_pct.mean():.4f}%')
axes[2].set_ylabel('Effective Spread (%)')
axes[2].set_xlabel('Time')
axes[2].set_title('Effective Spread (Trade price vs Mid)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Volume Profile Features

Analyze volume patterns and VWAP-based features.

In [ ]:
# VWAP (Volume-Weighted Average Price)
cumulative_volume = trade_data['volume'].cumsum()
cumulative_value = (trade_data['price'] * trade_data['volume']).cumsum()
vwap = cumulative_value / cumulative_volume

# Distance from VWAP
vwap_distance = (prices.reindex(trade_data.index, method='nearest') - vwap) / vwap * 100

# Volume rate of change
volume_roc = trade_data['volume'].pct_change(periods=10)

# Volume momentum
volume_momentum = trade_data['volume'].rolling(window=20).mean() / trade_data['volume'].rolling(window=50).mean()

# Relative volume (current vs average)
avg_volume = trade_data['volume'].rolling(window=100).mean()
relative_volume = trade_data['volume'] / avg_volume

print('Volume Profile Features Calculated:')
print('=' * 60)
print(f'VWAP: ${vwap.iloc[-1]:.2f}')
print(f'Current price vs VWAP: {vwap_distance.iloc[-1]:.4f}%')
print(f'Volume momentum range: {volume_momentum.min():.3f} to {volume_momentum.max():.3f}')
print(f'Relative volume range: {relative_volume.min():.3f} to {relative_volume.max():.3f}')

In [ ]:
# Visualize volume features
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Price vs VWAP
axes[0].plot(prices.index, prices, label='Price', linewidth=1.5)
axes[0].plot(vwap.index, vwap, label='VWAP', linewidth=2, linestyle='--', color='red')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Price vs Volume-Weighted Average Price (VWAP)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distance from VWAP
axes[1].plot(vwap_distance.index, vwap_distance, linewidth=1, color='purple')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].fill_between(vwap_distance.index, 0, vwap_distance,
                      where=(vwap_distance > 0), alpha=0.3, color='green', label='Above VWAP')
axes[1].fill_between(vwap_distance.index, 0, vwap_distance,
                      where=(vwap_distance < 0), alpha=0.3, color='red', label='Below VWAP')
axes[1].set_ylabel('Distance from VWAP (%)')
axes[1].set_title('Price Distance from VWAP')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Relative volume
axes[2].plot(relative_volume.index, relative_volume, linewidth=1, color='blue')
axes[2].axhline(1.0, color='r', linestyle='--', alpha=0.5, label='Average')
axes[2].fill_between(relative_volume.index, 1.0, relative_volume,
                      where=(relative_volume > 1.0), alpha=0.3, color='green')
axes[2].set_ylabel('Relative Volume')
axes[2].set_xlabel('Time')
axes[2].set_title('Relative Volume (Current / 100-period average)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Time-Based Features

Create lag features and rolling statistics for predictive modeling.

In [ ]:
# Returns at different horizons
returns_1 = prices.pct_change(1)
returns_5 = prices.pct_change(5)
returns_10 = prices.pct_change(10)
returns_20 = prices.pct_change(20)

# Lag features
price_lag_1 = prices.shift(1)
price_lag_5 = prices.shift(5)
price_lag_10 = prices.shift(10)

# Rolling statistics
price_sma_20 = prices.rolling(window=20).mean()
price_sma_50 = prices.rolling(window=50).mean()
price_std_20 = prices.rolling(window=20).std()
price_std_50 = prices.rolling(window=50).std()

# Momentum features
momentum_5 = prices - prices.shift(5)
momentum_10 = prices - prices.shift(10)
momentum_20 = prices - prices.shift(20)

# Acceleration
acceleration = returns_1.diff()

# High-low range
high_20 = prices.rolling(window=20).max()
low_20 = prices.rolling(window=20).min()
range_20 = high_20 - low_20
price_position = (prices - low_20) / range_20  # Position in range (0-1)

print('Time-Based Features Calculated:')
print('=' * 60)
print(f'Returns (1-period): mean={returns_1.mean():.6f}, std={returns_1.std():.6f}')
print(f'Returns (20-period): mean={returns_20.mean():.6f}, std={returns_20.std():.6f}')
print(f'Price position in 20-period range: {price_position.mean():.3f}')
print(f'20-period volatility: {price_std_20.mean():.4f}')

In [ ]:
# Visualize time-based features
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Price with moving averages
axes[0].plot(prices.index, prices, label='Price', linewidth=1.5, alpha=0.8)
axes[0].plot(price_sma_20.index, price_sma_20, label='SMA(20)', linewidth=1.5, linestyle='--')
axes[0].plot(price_sma_50.index, price_sma_50, label='SMA(50)', linewidth=1.5, linestyle='--')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Price with Moving Averages')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Momentum indicators
axes[1].plot(momentum_5.index, momentum_5, label='Momentum(5)', linewidth=1, alpha=0.7)
axes[1].plot(momentum_10.index, momentum_10, label='Momentum(10)', linewidth=1, alpha=0.7)
axes[1].plot(momentum_20.index, momentum_20, label='Momentum(20)', linewidth=1, alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('Momentum ($)')
axes[1].set_title('Price Momentum at Different Horizons')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Rolling volatility
axes[2].plot(price_std_20.index, price_std_20, label='Std(20)', linewidth=1.5)
axes[2].plot(price_std_50.index, price_std_50, label='Std(50)', linewidth=1.5)
axes[2].set_ylabel('Standard Deviation ($)')
axes[2].set_xlabel('Time')
axes[2].set_title('Rolling Volatility')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Feature Summary and Correlation

Combine all features and analyze their relationships.

In [ ]:
# Create feature DataFrame
features = pd.DataFrame({
    # Price features
    'price': prices,
    'returns_1': returns_1,
    'returns_5': returns_5,
    'returns_20': returns_20,
    
    # Technical indicators
    'rsi': rsi,
    'macd': macd_line,
    'bb_width': bb_upper - bb_lower,
    
    # Liquidity features
    'spread_bps': spread_bps,
    'total_depth': total_depth,
    'depth_imbalance': depth_imbalance,
    
    # Momentum
    'momentum_10': momentum_10,
    'momentum_20': momentum_20,
    
    # Volatility
    'volatility_20': price_std_20,
    
    # Volume
    'volume': trade_data['volume'].reindex(prices.index, method='nearest'),
}).dropna()

print('Feature Summary:')
print('=' * 80)
print(f'Total features: {len(features.columns)}')
print(f'Total observations: {len(features)}')
print('\nFeature statistics:')
print(features.describe())

In [ ]:
# Calculate and visualize correlation matrix
correlation_matrix = features.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Find highly correlated features
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.7:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print('\nHighly Correlated Features (|corr| > 0.7):')
    print('=' * 60)
    for feat1, feat2, corr in high_corr_pairs:
        print(f'{feat1:20s} <-> {feat2:20s}: {corr:+.3f}')

## 8. Feature Importance Analysis

Analyze which features are most predictive of future returns.

In [ ]:
# Calculate forward returns as target
forward_returns = features['price'].pct_change(5).shift(-5)

# Calculate correlation with forward returns
feature_importance = pd.DataFrame({
    'feature': [col for col in features.columns if col != 'price'],
    'correlation': [features[col].corr(forward_returns) for col in features.columns if col != 'price']
})
feature_importance['abs_correlation'] = feature_importance['correlation'].abs()
feature_importance = feature_importance.sort_values('abs_correlation', ascending=False)

print('Feature Importance for Predicting 5-Period Forward Returns:')
print('=' * 60)
print(feature_importance.to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(12, 6))
colors = ['green' if x > 0 else 'red' for x in feature_importance['correlation']]
plt.barh(feature_importance['feature'], feature_importance['correlation'], color=colors, alpha=0.7)
plt.xlabel('Correlation with 5-Period Forward Returns')
plt.title('Feature Importance Analysis')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we've created a comprehensive set of features for trading models:

### Feature Categories:

**1. Technical Indicators**
- RSI: Momentum oscillator identifying overbought/oversold conditions
- MACD: Trend-following momentum indicator
- Bollinger Bands: Volatility bands for mean reversion signals

**2. Liquidity Features**
- Bid-ask spreads: Transaction cost proxy
- Market depth: Available liquidity at different levels
- Amihud illiquidity: Price impact measure
- Depth imbalance: Order book asymmetry

**3. Microstructure Features**
- Order flow imbalance: Buy/sell pressure
- Trade intensity: Trading activity level
- Effective spread: Realized transaction costs
- Price impact: Market impact of trades

**4. Volume Features**
- VWAP: Volume-weighted benchmark price
- Relative volume: Current vs historical volume
- Volume momentum: Volume trend

**5. Time-Based Features**
- Returns at multiple horizons
- Moving averages and momentum
- Rolling volatility
- Lag features

### Key Takeaways:
- Feature engineering is crucial for building effective trading models
- Different feature categories capture different market dynamics
- Correlation analysis helps identify redundant features
- Feature importance guides model development priorities

### Next Steps:
- Use features in machine learning models for prediction
- Perform feature selection to reduce dimensionality
- Test features in backtesting frameworks
- Explore risk analysis techniques (notebook 05)